In [1]:
import numpy as np
import random
import pandas as pd
import time

### Funções Derivation Evolution

In [ ]:
# 1: Função Esférica 
def sphere (x: np. ndarray ) -> float :
    return np. sum(x **2) # Potência de 2

# 2: Função Ackley 
def ackley(x):    
    n = len(x)    
    return (-20*np.exp(-0.2*np.sqrt(np.sum(x**2)/n)) - np.exp(np.mean(np.cos(2*np.pi*x))) + 20 + np.e) 

# 3: Função Rastrigin 
def rastrigin(x):     
    n = len(x)     
    return 10*n + np.sum(x**2 - 10*np.cos(2*np.pi*x))

# rng = np.random.default_rng(42)
def cria_individuos (n, limites, rng):
    """ Os valores especificados no pdf. n=2, 5 e 10 

        limites=[-5.12,5.12] # para função esferica

        rgng é o seed para garantir a reprodutibilidade
    """
    individuo = rng.uniform(limites[0], limites[1], n) # A distribuição uniforme dá a mesma probabilidade de sortear qualquer valor entre -5 e 5
    return individuo

def criar_populacao(tamanho, n, limites, rng): #— gera N indivíduos.
    populacao = []
    for i in range (tamanho):
        populacao.append(cria_individuos(n, limites, rng))
    populacao = np.array(populacao)
    return populacao

def escolher(populacao, rng, x=3): # O torneio busca por valores mais proximos de zero.
    """ x: é número individuos selecionados par criar o try
    retorna os três selecionados a, b e c.
    """
    sorteados = rng.choice(populacao, x, replace=False) # replace=False, faz com que os elementos sejam obrigatoriamente diferente

    return sorteados

def mutacao (sorteados, f):
    ''' f = deve ser de 0 a 1
    '''
    vi = sorteados[0] + (f * (sorteados[1]-sorteados[2]))
    return vi
def escolher_target(populacao, sorteados,rng):
    posicao_t = rng.choice(len(populacao), 1, replace=False)
    target = populacao[posicao_t]
    flag = True
    if target not in (sorteados): return target, posicao_t

    else:
        while flag == True:
            posicao_t = rng.choice(len(populacao), 1, replace=False)
            target = populacao[posicao_t]
            if target not in (sorteados): return target, posicao_t

def crossover(target, vi, cr, rng): # target, vi, cr
    ''' vi = é a saída da mutação;
        cr é a probabilidade de copiar o elemento do verto mutante
    '''
    trial =[]
    n = len(vi)
    target = target[0]
    flag = False
    cont_target = 0
    # print(target)
    pb = rng.random((1,n))
    for i in range (len(vi)):
        if cr > pb[0][i]:
            trial.append(vi[i])
            flag = True
            # print(vi[i])
        else:
            # print(i, target[i])
            trial.append(target[i])
            # cont_target += 1
            
    if flag == False:    #garante que temha pelo menos uma posicão do vetor mutante
        posicao = random.randint(0,len(trial)-1)
        trial[posicao] = vi[posicao]
        # print('Flag False')

    trial = np.reshape(trial, (1,-1)) # transforma em uma martiz 1x1

    return trial

# avaliar 
def avaliar(funcao_objetivo, populacao): # o nfe_atual deverá ser revisado!
    fitness = np.apply_along_axis(funcao_objetivo,  axis= 1, arr=populacao)
    
    return fitness 

# seleção (elitista) 
def torneio(fitness, posicao, populacao, trail, fitness_vetor_mutante): # O torneio busca por valores mais proximos de zero.

    if fitness_vetor_mutante <=  fitness[posicao]:
        # print('if',fitness_vetor_mutante, ' menor que' , fitness[posicao])
        return trail #, fitness_vetor_mutante
    else: 
        # print('else',fitness_vetor_mutante, ' maior que' ,fitness[posicao])
        return populacao[posicao] #, fitness[posicao]



In [ ]:
    # Restrições 
def restricoes(x, funcao):
    g1 = np.sum(x) - 5     
    if funcao == 'esfera': g2 = np.sum(x**2) - 25     
    elif funcao == 'ackley': g2 = np.sum(x**2) - 50     
    elif funcao == 'rastrigin': g2 = np.sum(np.cos(x)) - 1     
    else: raise ValueError('Função desconhecida')    
    return np.array([g1, g2])

# Penalização 
def valor_penalizado(x, funcao, rho):
    f = {'esfera': sphere, 'ackley': ackley, 'rastrigin': rastrigin}[funcao](x)     
    g = restricoes(x, funcao)     
    return f + rho*np.sum(np.maximum(0, g)**2)

### ALGORITMMO DE

In [ ]:
def de(seed, tamanho, n, limites, f, cr, f_o, nfe_inicio, nfe_max): 
    rng = np.random.default_rng(seed)
    historico_convergencia = []

    populacao =criar_populacao(tamanho, n, limites, rng) #tamanho, n, limites, rng
    # escolhidos = escolher(populacao, rng, 3)
    # vi = mutacao (escolhidos, 0.5)

    nfe = nfe_inicio
    while (nfe <= nfe_max) or fitness_min <0:
        i = 0
        nova_populacao = []
        print(nfe)
        while i < tamanho:
            escolhidos = escolher(populacao, rng, 3)
            vi = mutacao (escolhidos, f)

            aux = escolher_target(populacao,escolhidos,rng)
            posicao = aux[1]
            target = aux[0]

            trial = crossover(target, vi, cr,rng)
            f_target = avaliar(f_o, populacao)
            f_trail = avaliar(f_o, trial)
#           vencedor = torneio(f_target, posicao, populacao, trial, f_trail  )
            vencedor = torneio(f_target, posicao, populacao, trial, f_trail  )

            nova_populacao.append(vencedor[0])
                      
            i = len(nova_populacao) 

        nova_populacao = np.array(nova_populacao)[:tamanho]  
        fitness = avaliar(f_o,nova_populacao)      
        historico_convergencia.append(fitness.min())
        fitness_min =fitness.min()
        nfe_atual = nfe + len(nova_populacao) # se for preciso
        nfe =  nfe_atual
        populacao = nova_populacao

    
    posicao = np.argmin(fitness)
    melhor_solucao = populacao[posicao]

    melhor_fitness = fitness[posicao]
    
    return populacao, melhor_solucao, historico_convergencia, melhor_fitness, nfe

### Testes das funções

2. criar_populacao(tamanho, n, limites, rng): # retorna uma população de tamanho == tamanho
    1. cria_individuos (n, limites, rng): # retorna um individuo
3. escolher(populacao, rng, x=3): # retorna tres individuos diferentes: a, b e c
4. mutacao (sorteados, f): # retorna um array 1d
5. escolher_target(populacao, sorteados): # retorna um array com o target e um com a posição
6. crossover(target, vi, cr): # retorna o trial
8. avaliar(funcao_objetivo, populacao): # retorna um array com o tamanho da população
    7. sphere (x: np. ndarray ) -> float: # retorna o fitnees individual
9. torneio(fitness, posicao, população, trail, fitness_vetor_mutante, rng): # retorna um array e uma posição

In [148]:
seed = 42
tamanho = 10
n = 2
limites = [-5.12, 5.12]
f = 0.4
cr = 0.5
f_o = sphere
nfe_inicio = 0
nfe_max = 500


In [ ]:
teste = de(seed, tamanho, n, limites, f, cr, f_o, nfe_inicio, nfe_max)